In [ ]:
# imports, nødvendige for å kjøre koden 

import os
import json
import logging
import numpy as np
import pandas as pd
import oracledb
from google.cloud import secretmanager
from google.cloud.bigquery import Client

# Trenges når jeg kjører lokal i venv, da bruker jeg mine creds. 
# installer "uv pip install python-dotenv" og opprett en .env fil
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Skriver forstårlig og menneskelig logging --> tid - log level (info, warning etc) - melding

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

In [ ]:
# Henting og håndtering av hemmeligheter. Henter fra GSM og setter dem som miljø variabler

def set_secrets_as_envs():

    logger.info("Loading secrets from Secret Manager")

    client = secretmanager.SecretManagerServiceClient()
    resource_name = f"{os.environ['KNADA_TEAM_SECRET']}/versions/latest"
    response = client.access_secret_version(name=resource_name)
    secret_payload = response.payload.data.decode("UTF-8")
    secrets = json.loads(secret_payload)
    os.environ.update(secrets)
    
    
def get_secrets():

    env = os.getenv("ENV", "prod")

    logger.info(f"Running in environment: {env}")

    set_secrets_as_envs()

    secrets = {
        "user": os.getenv("DB_USER"),
        "password": os.getenv("DB_PASSWORD"),
        "host": os.getenv("DBT_ORCL_HOST"),
        "service": os.getenv("DBT_ORCL_SERVICE"),
        "project_id": os.getenv("BS_PROJECT_ID"),
        "table_uri": os.getenv("TS_v2_TABLE_URI"),
    }

     # Overskriver bruker og passord fra gsm. bruker personlige creds istedet
    if env == "local":

        logger.info("Overriding Oracle credentials with local user")

        secrets["user"] = os.getenv("ORACLE_USER")
        secrets["password"] = os.getenv("ORACLE_PASSWORD")

    return secrets

In [ ]:
# Henter max verdien i kolonnen endret_tid fra oracle tabellen, trenges for en delta last!

def get_latest_endret_tid(cursor):

    logger.info("Fetching max(endret_tid) from Oracle")

    sql = """select max(endret_tid) from fam_ts_meta_data_v2"""
    cursor.execute(sql)
    max_value = cursor.fetchone()[0]

    logger.info(f"latest endret_tid: {max_value}")

    return max_value

In [ ]:

def get_data_from_table(secrets, endret_tid=None):

    logger.info("Fetching data from BigQuery")

    client = Client(project=secrets["project_id"])

    if endret_tid:
        query = f"""
        SELECT *
        FROM `{secrets['table_uri']}`
        WHERE endret_tid >= TIMESTAMP_SUB(TIMESTAMP('{endret_tid}'), INTERVAL 1 DAY)
        """
    else:
        logger.info("Full load")
        query = f"SELECT * FROM `{secrets['table_uri']}`"

    # Beholder den nyeste raden basert på ekstern_behandling_id, 
    # det vil si hvis innholdet for en rad med samme ekstern_behandling_id har forandret seg fra dag til dag
    # tar vi vare kun på den nyeste
    df = (
        client.query(query)
        .to_dataframe()
        .sort_values("endret_tid", ascending=True)
        .drop_duplicates(subset=["ekstern_behandling_id"], keep="last")
)

    logger.info(f"Rows fetched: {len(df)}")

    return df

In [ ]:
def transform_to_raw(df):

    logger.info("Transforming to RAW format")

    raw_df = pd.DataFrame({
        "json_str": df.apply(
            lambda row: json.dumps(
                row.to_dict(),
                ensure_ascii=False,
                default=str
            ),
            axis=1
        ),
        "ekstern_behandling_id": df["ekstern_behandling_id"],
        "opprettet_tid": df["opprettet_tid"],
        "endret_tid": df["endret_tid"]
    })

    return raw_df

In [ ]:
# Kilden kjører en hel last hver natt. Tabellen kan slettes og re-bygges ved en endring (eks, vårt krav om å legge til en order by på opprettet tid) 
# derfor legger jeg noe kode for validering av datasettet/tabellen

def validate_dataframe(df):

    logger.info("Validering av datasettet")

    required_columns = ["json_str", "ekstern_behandling_id", "opprettet_tid"]

    missing_columns = [
        col for col in required_columns if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}")

    if df.empty:
        raise ValueError("DataFrame is empty")

    logger.info("Validering compeleted!")

In [ ]:
#### L delen av ETL jobben (Loading)

# lager og returnerer en oracle connection
def create_connection(secrets):

    logger.info("Åpner en connection mot Oracle")

    dsn = oracledb.makedsn(secrets["host"], 1521, service_name=secrets["service"])
    user = f"{secrets['user']}[DVH_FAM_EF]"
    conn = oracledb.connect(user=user, password=secrets["password"], dsn=dsn)

    return conn

    
def load_dataframe(cursor, df):

    logger.info("Loading into Oracle")

    insert_sql = """
        INSERT INTO fam_ts_meta_data_v2 (
            melding,
            ekstern_behandling_id,
            opprettet_tid,
            endret_tid
        )
        VALUES (:1, :2, :3, :4)
    """

    rows = [
        tuple(
            None if pd.isna(v)
            else v.item() if hasattr(v, "item")
            else v
            for v in row
        )
        for row in df.itertuples(index=False, name=None)
    ]

    cursor.executemany(insert_sql, rows)

    logger.info(f"Inserted rows: {len(rows)}")

In [ ]:
def send_data():

    logger.info("Starting pipeline")

    secrets = get_secrets()

    with create_connection(secrets) as conn:
        try:
            with conn.cursor() as cursor:

                endret_tid = get_latest_endret_tid(cursor)

                df = get_data_from_table(secrets, endret_tid)

                if df.empty:
                    logger.info("No new data found")
                    return

                df = transform_to_raw(df)

                validate_dataframe(df)

                load_dataframe(cursor, df)

                conn.commit()

                logger.info("Pipeline completed successfully")

        except Exception:
            logger.exception("Pipeline failed")
            conn.rollback()
            raise

In [ ]:
send_data()